# Lab 03-05: Model Selection from Hub| | ||---|---|| **Module** | 03 -- Application Development (30% of exam) || **Estimated Questions** | ~13 questions || **UI Sections** | Discover, Marketplace, Serving || **Estimated Time** | 35--45 minutes || **Difficulty** | Beginner-Intermediate |---## What Is Model Selection from Hub and Why Does It Matter?The Databricks platform gives you **three distinct places** to find and deployfoundation models:1. **Foundation Model APIs** -- Pre-deployed models available instantly via   pay-per-token endpoints (DBRX, Llama, Mixtral, etc.)2. **Marketplace** -- A catalog of third-party and open-source models you can   browse, compare, and deploy to your own serving endpoints3. **AI Gateway (External Models)** -- A proxy layer that lets you call   external providers (OpenAI, Anthropic, Cohere) through Databricks with   unified governance and rate limitingThe exam tests whether you know **where** to find models, **how** to comparethem using model cards and benchmarks, and **when** to use each approach.Getting this wrong means either overpaying for a model that is too powerful foryour task, or deploying one that cannot handle it.---## Mental Model> **Analogy:** The Marketplace is like an app store for AI models. Foundation> Model APIs are the pre-installed apps -- always available, no setup, you just> use them. Marketplace models are third-party apps you install -- more variety,> but you deploy them yourself and manage the serving endpoint. External models> (via AI Gateway) are like web apps you access through a browser -- the model> lives somewhere else (OpenAI, Anthropic), but Databricks provides a secure> gateway so you get logging, rate limiting, and unified access control.---## Exam Alert -- Common Traps| Trap | Why Students Fall For It | The Truth ||---|---|---|| "All models are available via Foundation Model APIs" | Foundation Model APIs are convenient, so students assume everything is there | Only select models (DBRX, Llama, Mixtral, etc.) are pre-deployed. Others require Marketplace or custom deployment. || "Marketplace models auto-deploy when you install them" | The Marketplace "Get" button feels like installing an app | You must **create a serving endpoint** for Marketplace models. Getting a model only adds it to your catalog. || "External models (OpenAI, Anthropic) are in the Marketplace" | Both are places to "get" models | External models use **AI Gateway** (External Models in Serving), not Marketplace. Marketplace is for models you deploy on Databricks compute. || "Bigger models are always better" | More parameters = smarter, right? | Bigger models are slower and more expensive. A 7B model may outperform a 70B model on a narrow task if fine-tuned. Match model size to task complexity. || "Model benchmarks tell you everything" | MMLU scores are prominently displayed | Benchmarks test general ability. Your specific task may need domain evaluation. Use benchmarks for shortlisting, then evaluate on your own data. |---## Prerequisites- Completed Lab 03-04 (Embedding Model Selection)- Familiarity with the OpenAI-compatible SDK pattern---## Exam Objectives Covered- Navigate Discover and Marketplace to find models- Compare models using model cards, benchmarks, and metadata- Understand Foundation Model APIs vs Marketplace vs External Models- Know when to use each model sourcing approach

---## Setup

In [ ]:
# ------------------------------------------------------------------
# Setup: Install dependencies
# ------------------------------------------------------------------

%pip install openai --quiet

# Restart Python to pick up the new package
dbutils.library.restartPython()


**Expected output:** Installation logs followed by `Python interpreter will be restarted.`

In [ ]:
# ------------------------------------------------------------------
# Setup: Initialize the OpenAI-compatible client for Databricks
# ------------------------------------------------------------------

from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ.get("DATABRICKS_TOKEN"),
    base_url=f"{os.environ.get('DATABRICKS_HOST')}/serving-endpoints"
)

MODEL = "databricks-meta-llama-3-3-70b-instruct"


def chat(user_msg, system_msg="You are a helpful assistant.", temperature=0.0, max_tokens=512):
    """Send a chat completion request and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


print("Client ready.  Model:", MODEL)


**Expected output:**```Client ready.  Model: databricks-meta-llama-3-3-70b-instruct```

---## Step 1: Navigating Discover and MarketplaceDatabricks provides two UI surfaces for finding models. Understanding thedifference is essential for the exam.### The Discover Page**UI -->** Left navigation bar --> click **Discover** (or **Serving** --> **Foundation Models**)The Discover page shows **Foundation Model APIs** -- models that Databrickshas pre-deployed and makes available as pay-per-token endpoints. You do NOTneed to create a serving endpoint for these models. They are ready to useimmediately.**What you see on the Discover page:**- Model name (e.g., `databricks-meta-llama-3-3-70b-instruct`)- Provider (Meta, Databricks, Mistral, etc.)- Task type (Chat, Completion, Embeddings)- Context window length- A "Use" button that shows you the API call syntax### The Marketplace**UI -->** Left navigation bar --> click **Marketplace**The Marketplace is a broader catalog of models from various providers. UnlikeFoundation Model APIs, Marketplace models are NOT pre-deployed. You browse,compare, and then deploy them to your own serving endpoint.**What you see on a Marketplace model card:**- Model description and use cases- License (commercial use allowed?)- Parameter count- Benchmark scores (MMLU, HumanEval, MT-Bench)- Provider information- "Get" button -- adds the model to your catalog (does NOT deploy it)### Key Difference```FOUNDATION MODEL APIs                    MARKETPLACE========================                 ========================Pre-deployed by Databricks               You deploy to your endpointPay-per-token pricing                    You pay for computeLimited selection (curated)              Broad selection (open catalog)No setup required                        Requires serving endpoint setupAlways available                         Available after deployment```> **Exam tip:** If a question asks about models that are "immediately available"> or "require no deployment," the answer is Foundation Model APIs. If it asks> about "browsing a catalog" or "deploying a third-party model," that is Marketplace.

---## Step 2: Foundation Model APIs -- List Available Models and Call OneFoundation Model APIs are the fastest way to use an LLM on Databricks. Let'sexplore what is available and make a call.

In [ ]:
# ------------------------------------------------------------------
# Step 2a: List available Foundation Model API endpoints
# ------------------------------------------------------------------
# Foundation Model APIs appear as serving endpoints with the
# "FOUNDATION_MODEL_API" type. We can list them via the REST API.

import requests
import os

host = os.environ.get("DATABRICKS_HOST")
token = os.environ.get("DATABRICKS_TOKEN")

headers = {"Authorization": f"Bearer {token}"}
response = requests.get(f"{host}/api/2.0/serving-endpoints", headers=headers)

if response.status_code == 200:
    endpoints = response.json().get("endpoints", [])

    # Filter for Foundation Model API endpoints
    fmapi_endpoints = [
        ep for ep in endpoints
        if ep.get("endpoint_type") == "FOUNDATION_MODEL_API"
        or ep.get("name", "").startswith("databricks-")
    ]

    print("=" * 70)
    print("FOUNDATION MODEL API ENDPOINTS")
    print("=" * 70)
    for ep in sorted(fmapi_endpoints, key=lambda x: x["name"]):
        name = ep["name"]
        state = ep.get("state", {}).get("ready", "UNKNOWN")
        print(f"  {name:<50s}  [{state}]")
    print(f"\nTotal Foundation Model API endpoints: {len(fmapi_endpoints)}")
else:
    print(f"Could not list endpoints (status {response.status_code})")
    print("This is expected if running outside Databricks.")
    print("\nCommon Foundation Model API endpoints include:")
    print("  databricks-meta-llama-3-3-70b-instruct   (Chat)")
    print("  databricks-meta-llama-3-1-405b-instruct  (Chat)")
    print("  databricks-dbrx-instruct                 (Chat)")
    print("  databricks-mixtral-8x7b-instruct         (Chat)")
    print("  databricks-bge-large-en                  (Embeddings)")
    print("  databricks-gte-large-en                  (Embeddings)")


**Expected output:**```======================================================================FOUNDATION MODEL API ENDPOINTS======================================================================  databricks-bge-large-en                             [READY]  databricks-dbrx-instruct                            [READY]  databricks-gte-large-en                             [READY]  databricks-meta-llama-3-1-405b-instruct             [READY]  databricks-meta-llama-3-3-70b-instruct              [READY]  databricks-mixtral-8x7b-instruct                    [READY]Total Foundation Model API endpoints: 6```Note: The exact list varies by workspace and region. The key takeaway is thatthese endpoints are **pre-deployed and ready** -- no setup required.

In [ ]:
# ------------------------------------------------------------------
# Step 2b: Call a Foundation Model API endpoint
# ------------------------------------------------------------------
# This is the same pattern from earlier labs. Foundation Model APIs
# use the OpenAI-compatible chat completions format.

result = chat(
    user_msg="Explain what a foundation model is in exactly 2 sentences.",
    temperature=0.0,
    max_tokens=100
)

print("=== Foundation Model API Call ===")
print(f"Model:    {MODEL}")
print(f"Response: {result}")


**Expected output:**```=== Foundation Model API Call ===Model:    databricks-meta-llama-3-3-70b-instructResponse: A foundation model is a large-scale AI model trained on broaddata that can be adapted to a wide range of tasks. It serves as ageneral-purpose base that can be fine-tuned or prompted for specificapplications like text generation, classification, and question answering.```**What just happened:** We called a Foundation Model API endpoint with zerosetup -- no deployment, no endpoint creation, no compute provisioning. This isthe simplest path from "I need an LLM" to "I have an LLM response." The examexpects you to know that Foundation Model APIs provide this frictionlessexperience.

---## Step 3: Understanding Model Cards and Comparing ModelsWhether you find a model on Discover, Marketplace, or Hugging Face, the**model card** is your primary tool for evaluation. The exam tests whether youcan read a model card and make the right selection.### Key Fields on a Model Card| Field | What It Tells You | Why It Matters ||---|---|---|| **Task type** | Chat, completion, embeddings, code generation | Must match your use case || **Parameter count** | 7B, 13B, 70B, 405B | Larger = more capable but slower and more expensive || **Context window** | 4K, 8K, 32K, 128K tokens | Must fit your input (document + prompt + history) || **License** | Apache 2.0, Llama license, proprietary | Must allow commercial use for production apps || **Benchmarks** | MMLU, HumanEval, MT-Bench, etc. | Quantifies general capability (see below) || **Fine-tuning base** | What data/task it was tuned on | Domain-specific tuning may match your needs |### Key Benchmarks to Know for the Exam| Benchmark | What It Measures | Score Range | Use When Selecting For ||---|---|---|---|| **MMLU** | General knowledge across 57 academic subjects | 0--100% | General-purpose question answering || **HumanEval** | Code generation (Python) | 0--100% | Code assistant applications || **MT-Bench** | Multi-turn conversation quality | 1--10 | Chatbot applications || **HellaSwag** | Commonsense reasoning | 0--100% | Reasoning-heavy tasks || **TruthfulQA** | Resistance to generating false statements | 0--100% | Factual accuracy matters |> **Exam tip:** If a question asks "which benchmark evaluates code generation?"> the answer is **HumanEval**. For "general knowledge," it is **MMLU**. For> "conversation quality," it is **MT-Bench**.

In [ ]:
# ------------------------------------------------------------------
# Step 3: Compare models side-by-side using key selection criteria
# ------------------------------------------------------------------
# This is the kind of comparison table you would build when choosing
# a model for a production application.

MODEL_CATALOG = [
    {
        "name": "Meta Llama 3.3 70B Instruct",
        "endpoint": "databricks-meta-llama-3-3-70b-instruct",
        "params": "70B",
        "context": "128K",
        "license": "Llama 3.3 Community",
        "source": "Foundation Model API",
        "mmlu": 86.0,
        "humaneval": 88.4,
        "mt_bench": 8.8,
        "best_for": "General-purpose chat, reasoning, instruction following",
    },
    {
        "name": "Meta Llama 3.1 405B Instruct",
        "endpoint": "databricks-meta-llama-3-1-405b-instruct",
        "params": "405B",
        "context": "128K",
        "license": "Llama 3.1 Community",
        "source": "Foundation Model API",
        "mmlu": 88.6,
        "humaneval": 89.0,
        "mt_bench": 9.1,
        "best_for": "Hardest reasoning tasks, highest quality when cost is not a concern",
    },
    {
        "name": "DBRX Instruct",
        "endpoint": "databricks-dbrx-instruct",
        "params": "132B (MoE, 36B active)",
        "context": "32K",
        "license": "Databricks Open Model License",
        "source": "Foundation Model API",
        "mmlu": 73.7,
        "humaneval": 70.1,
        "mt_bench": 8.0,
        "best_for": "Cost-effective general tasks, Databricks-native",
    },
    {
        "name": "Mixtral 8x7B Instruct",
        "endpoint": "databricks-mixtral-8x7b-instruct",
        "params": "47B (MoE, 13B active)",
        "context": "32K",
        "license": "Apache 2.0",
        "source": "Foundation Model API",
        "mmlu": 70.6,
        "humaneval": 40.2,
        "mt_bench": 7.6,
        "best_for": "Fast, cost-effective tasks where Apache license is needed",
    },
    {
        "name": "BGE Large EN",
        "endpoint": "databricks-bge-large-en",
        "params": "335M",
        "context": "512 tokens",
        "license": "MIT",
        "source": "Foundation Model API",
        "mmlu": "N/A (embedding)",
        "humaneval": "N/A (embedding)",
        "mt_bench": "N/A (embedding)",
        "best_for": "Text embeddings for vector search / RAG",
    },
]

print("=" * 90)
print("MODEL COMPARISON -- FOUNDATION MODEL APIs")
print("=" * 90)
print(f"{'Model':<35s} {'Params':<22s} {'Context':<10s} {'MMLU':<8s} {'Source'}")
print("-" * 90)
for m in MODEL_CATALOG:
    mmlu = f"{m['mmlu']:.1f}" if isinstance(m['mmlu'], float) else m['mmlu']
    print(f"{m['name']:<35s} {m['params']:<22s} {m['context']:<10s} {mmlu:<8s} {m['source']}")

print()
print("=" * 90)
print("DETAILED VIEW")
print("=" * 90)
for m in MODEL_CATALOG:
    print(f"\n  {m['name']}")
    print(f"  {'~' * len(m['name'])}")
    print(f"  Endpoint:  {m['endpoint']}")
    print(f"  Params:    {m['params']}")
    print(f"  Context:   {m['context']}")
    print(f"  License:   {m['license']}")
    print(f"  Best for:  {m['best_for']}")


**Expected output:**```==========================================================================================MODEL COMPARISON -- FOUNDATION MODEL APIs==========================================================================================Model                               Params                 Context    MMLU     Source------------------------------------------------------------------------------------------Meta Llama 3.3 70B Instruct         70B                    128K       86.0     Foundation Model APIMeta Llama 3.1 405B Instruct        405B                   128K       88.6     Foundation Model APIDBRX Instruct                       132B (MoE, 36B active) 32K        73.7     Foundation Model APIMixtral 8x7B Instruct               47B (MoE, 13B active)  32K        70.6     Foundation Model APIBGE Large EN                        335M                   512 tokens N/A ...  Foundation Model API```**What just happened:** We built a comparison table of Foundation Model APImodels. In practice, you would do this same comparison when choosing betweenMarketplace models -- with a much larger catalog to search through.

---## Step 4: Deploying a Marketplace ModelWhen the Foundation Model APIs do not include the model you need -- perhaps youwant a domain-specific model, a newer release, or a model with a particularlicense -- you turn to the **Marketplace**.### The Marketplace Workflow```STEP 1: Browse                    STEP 2: Get                     STEP 3: Deploy========================          ========================         ========================UI --> Marketplace                Click "Get" on a model           UI --> Serving --> CreateBrowse by task, provider,         Model is added to your           Select the model fromlicense, popularity               Unity Catalog                    your catalog, configure                                  (NOT yet deployed!)              compute, create endpoint```### UI Walkthrough1. **UI -->** Left nav --> **Marketplace**2. Use filters: Task (Text Generation, Embeddings, etc.), Provider, License3. Click on a model card to see details, benchmarks, and documentation4. Click **"Get"** -- this adds the model artifact to your Unity Catalog5. **UI -->** Left nav --> **Serving** --> **Create serving endpoint**6. Select the model from your catalog7. Configure compute (GPU type, scaling, etc.)8. Click **Create** -- Databricks provisions the endpoint### Critical Exam Point> **Getting a model from Marketplace does NOT deploy it.** You must separately> create a serving endpoint. This is the #1 Marketplace trap on the exam.The "Get" button is like downloading an app -- it is on your device, but it isnot running yet. Creating a serving endpoint is like launching the app.

In [ ]:
# ------------------------------------------------------------------
# Step 4: Decision framework -- Foundation Model API vs Marketplace
# ------------------------------------------------------------------
# This interactive exercise helps you practice the decision logic
# that the exam tests.

SCENARIOS = [
    {
        "requirement": "Quick prototype -- need a chat model RIGHT NOW, no setup",
        "answer": "Foundation Model API",
        "why": (
            "Foundation Model APIs are pre-deployed. No endpoint creation needed. "
            "Pick one from Discover and start calling it immediately."
        ),
    },
    {
        "requirement": "Need a medical domain model fine-tuned on clinical notes",
        "answer": "Marketplace",
        "why": (
            "Domain-specific fine-tuned models are not in Foundation Model APIs. "
            "Search the Marketplace for medical/clinical models, get it, "
            "then deploy to a serving endpoint."
        ),
    },
    {
        "requirement": "Company policy requires using OpenAI GPT-4 for a specific task",
        "answer": "AI Gateway (External Models)",
        "why": (
            "GPT-4 is not a Databricks-hosted model. Use AI Gateway to configure "
            "an external model endpoint that proxies requests to OpenAI while "
            "providing Databricks governance, logging, and rate limiting."
        ),
    },
    {
        "requirement": "Cost-sensitive batch processing of 1M documents for classification",
        "answer": "Foundation Model API (pay-per-token) or Marketplace (provisioned throughput)",
        "why": (
            "For high volume, compare: Foundation Model API pay-per-token cost vs. "
            "deploying a smaller Marketplace model on dedicated compute. "
            "A smaller model (7B) on provisioned throughput may be cheaper at scale."
        ),
    },
    {
        "requirement": "Need to call Anthropic Claude for complex reasoning tasks",
        "answer": "AI Gateway (External Models)",
        "why": (
            "Claude is an external model (Anthropic). Configure it via AI Gateway "
            "in the Serving UI under External Models. This gives you unified "
            "access control, rate limiting, and usage tracking through Databricks."
        ),
    },
    {
        "requirement": "Want to compare 5 different open-source models on our evaluation dataset",
        "answer": "Marketplace + Foundation Model APIs",
        "why": (
            "Use Foundation Model APIs for models already available there. "
            "For others, get them from Marketplace and deploy. Then run your "
            "evaluation dataset against all endpoints to compare."
        ),
    },
]

print("=" * 76)
print("MODEL SOURCING DECISION FRAMEWORK")
print("=" * 76)

for i, s in enumerate(SCENARIOS, 1):
    print(f"\n{'~' * 76}")
    print(f"  Scenario {i}: {s['requirement']}")
    print(f"{'~' * 76}")
    print(f"  --> Use: {s['answer']}")
    print(f"  Why:  {s['why']}")

print(f"\n{'=' * 76}")
print("DECISION TREE SUMMARY")
print("=" * 76)
print()
print("  Need a model?")
print("  |")
print("  +-- Is it a Databricks Foundation Model (DBRX, Llama, Mixtral, BGE, GTE)?")
print("  |     YES --> Foundation Model API (no setup, pay-per-token)")
print("  |")
print("  +-- Is it an open-source / third-party model you want to host on Databricks?")
print("  |     YES --> Marketplace (browse, get, deploy to serving endpoint)")
print("  |")
print("  +-- Is it an external provider model (OpenAI, Anthropic, Cohere, etc.)?")
print("        YES --> AI Gateway / External Models (proxy through Databricks)")


**Expected output:**```============================================================================MODEL SOURCING DECISION FRAMEWORK============================================================================~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  Scenario 1: Quick prototype -- need a chat model RIGHT NOW, no setup~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  --> Use: Foundation Model API  Why:  Foundation Model APIs are pre-deployed. No endpoint creation needed. ...  ...============================================================================DECISION TREE SUMMARY============================================================================  Need a model?  |  +-- Is it a Databricks Foundation Model (DBRX, Llama, Mixtral, BGE, GTE)?  |     YES --> Foundation Model API (no setup, pay-per-token)  |  +-- Is it an open-source / third-party model you want to host on Databricks?  |     YES --> Marketplace (browse, get, deploy to serving endpoint)  |  +-- Is it an external provider model (OpenAI, Anthropic, Cohere, etc.)?        YES --> AI Gateway / External Models (proxy through Databricks)```

---## Step 5: External Models via AI GatewayAI Gateway allows you to call **external model providers** (OpenAI, Anthropic,Cohere, Amazon Bedrock, Google Vertex AI) through Databricks. The externalmodel runs on the provider's infrastructure, but Databricks acts as a proxythat adds governance, logging, rate limiting, and unified access control.### Why Use AI Gateway Instead of Calling the Provider Directly?| Benefit | What It Means ||---|---|| **Unified governance** | Same access controls (ACLs) as Databricks-hosted models || **Usage tracking** | All requests logged to inference tables for monitoring || **Rate limiting** | Prevent individual users/apps from exhausting your API quota || **Cost tracking** | Track spend per endpoint, per user, per application || **Failover** | Configure fallback routes if one provider is down || **Single SDK** | Same OpenAI-compatible SDK for Databricks models AND external models |### UI Walkthrough: Creating an External Model Endpoint1. **UI -->** Left nav --> **Serving**2. Click **Create serving endpoint**3. Under **Served entities**, select **External model**4. Choose the provider (OpenAI, Anthropic, etc.)5. Enter your API key (stored securely as a Databricks secret)6. Select the model (e.g., `gpt-4`, `claude-3-sonnet`)7. Configure rate limits and access permissions8. Click **Create**### Critical Exam Point> **External models are NOT in the Marketplace.** The Marketplace is for models> you deploy on Databricks compute. External models use AI Gateway, which proxies> requests to the external provider's infrastructure.

In [ ]:
# ------------------------------------------------------------------
# Step 5: Calling an External Model via AI Gateway
# ------------------------------------------------------------------
# Once configured, external model endpoints use the SAME OpenAI-
# compatible SDK. The only difference is the endpoint name.
#
# This is CONCEPTUAL code -- it requires an actual external model
# endpoint to be configured in your workspace.

# ---------- CONCEPTUAL CODE ----------
# If you had an external model endpoint called "openai-gpt4-gateway":
#
# response = client.chat.completions.create(
#     model="openai-gpt4-gateway",    # <-- endpoint name, not model name
#     messages=[
#         {"role": "system", "content": "You are a helpful assistant."},
#         {"role": "user", "content": "Explain quantum computing in one sentence."}
#     ],
#     temperature=0.0,
#     max_tokens=100
# )
# print(response.choices[0].message.content)

# ---------- KEY INSIGHT ----------
# The SDK call is IDENTICAL to calling a Foundation Model API endpoint.
# The only difference is the endpoint name. This is the power of AI Gateway:
# unified interface, regardless of where the model actually runs.

print("=" * 70)
print("AI GATEWAY -- EXTERNAL MODELS SUMMARY")
print("=" * 70)
print()
print("HOW IT WORKS:")
print("  Your Code  -->  Databricks AI Gateway  -->  External Provider")
print("  (OpenAI SDK)    (logging, rate limits,      (OpenAI, Anthropic,")
print("                   access control)             Cohere, etc.)")
print()
print("SETUP (UI):")
print("  Serving --> Create endpoint --> External model --> Pick provider")
print("  --> Enter API key --> Configure rate limits --> Create")
print()
print("CALLING IT (Code):")
print("  Same OpenAI-compatible SDK -- just use the gateway endpoint name.")
print('  client.chat.completions.create(model="my-gateway-endpoint", ...)')
print()
print("KEY EXAM FACTS:")
print("  - External models do NOT run on Databricks compute")
print("  - External models are NOT in the Marketplace")
print("  - AI Gateway provides governance for external model calls")
print("  - The SDK interface is the same for all model types")


**Expected output:**```======================================================================AI GATEWAY -- EXTERNAL MODELS SUMMARY======================================================================HOW IT WORKS:  Your Code  -->  Databricks AI Gateway  -->  External Provider  (OpenAI SDK)    (logging, rate limits,      (OpenAI, Anthropic,                   access control)             Cohere, etc.)SETUP (UI):  Serving --> Create endpoint --> External model --> Pick provider  --> Enter API key --> Configure rate limits --> CreateCALLING IT (Code):  Same OpenAI-compatible SDK -- just use the gateway endpoint name.  client.chat.completions.create(model="my-gateway-endpoint", ...)KEY EXAM FACTS:  - External models do NOT run on Databricks compute  - External models are NOT in the Marketplace  - AI Gateway provides governance for external model calls  - The SDK interface is the same for all model types```

---## Step 6: The Complete Model Selection Decision FrameworkNow let's bring everything together into a comprehensive decision frameworkthat covers all three sourcing approaches plus the model card evaluationcriteria.

In [ ]:
# ------------------------------------------------------------------
# Step 6: Complete Model Selection Decision Framework
# ------------------------------------------------------------------
# This is your exam reference -- the full decision process for
# selecting and sourcing a model on Databricks.

print("=" * 76)
print("COMPLETE MODEL SELECTION FRAMEWORK")
print("=" * 76)

print()
print("PHASE 1: DEFINE REQUIREMENTS")
print("=" * 40)
print("  1. What is the TASK? (chat, classification, code gen, embeddings, etc.)")
print("  2. What is the SCALE? (prototype vs production, requests/sec)")
print("  3. What is the BUDGET? (pay-per-token vs provisioned throughput)")
print("  4. What CONTEXT LENGTH do you need? (short prompts vs long documents)")
print("  5. Any LICENSE constraints? (Apache 2.0, commercial use, etc.)")
print("  6. Any COMPLIANCE requirements? (data residency, no external calls)")

print()
print("PHASE 2: SOURCE THE MODEL")
print("=" * 40)
print("  Requirement                          Source")
print("  -----------------------------------  --------------------------------")
print("  Pre-deployed, instant access         Foundation Model API")
print("  Specific open-source model           Marketplace --> Deploy to Serving")
print("  External provider (OpenAI, etc.)     AI Gateway (External Models)")
print("  Custom fine-tuned model              Upload to UC --> Deploy to Serving")

print()
print("PHASE 3: EVALUATE USING MODEL CARDS")
print("=" * 40)
print("  Criteria           What to Check               Red Flag")
print("  -----------------  --------------------------  ---------------------------")
print("  Task match         Model card 'use cases'      Model designed for different task")
print("  Benchmark scores   MMLU, HumanEval, MT-Bench   Scores significantly below peers")
print("  Context window     Token limit                 Input exceeds context window")
print("  License            Commercial use allowed?     Restrictive license for production")
print("  Parameter count    Size vs speed tradeoff      Over-provisioned for simple task")

print()
print("PHASE 4: TEST ON YOUR DATA")
print("=" * 40)
print("  - Benchmarks are general; YOUR task may differ")
print("  - Create an evaluation dataset with expected outputs")
print("  - Test 2-3 candidate models on YOUR data")
print("  - Compare: accuracy, latency, cost per request")
print("  - Pick the model that gives the best quality-to-cost ratio")

# --- Practical quiz ---
print()
print("=" * 76)
print("SELF-CHECK QUIZ: Model Selection Scenarios")
print("=" * 76)

QUIZ = [
    {
        "q": "You need embeddings for a RAG pipeline. Where do you get the model?",
        "a": "Foundation Model API (databricks-bge-large-en or databricks-gte-large-en)",
        "why": "Embedding models are available as Foundation Model APIs -- no deployment needed.",
    },
    {
        "q": "Your company requires Apache 2.0 license for all production models. Which chat model?",
        "a": "Mixtral 8x7B Instruct (Apache 2.0) via Foundation Model API",
        "why": "Llama uses Llama Community License. DBRX uses Databricks Open Model License. Mixtral is Apache 2.0.",
    },
    {
        "q": "You need to process documents with 100K+ token context. Which model?",
        "a": "Llama 3.3 70B or Llama 3.1 405B (128K context window)",
        "why": "DBRX and Mixtral have 32K context windows -- too small. Llama 3.x models support 128K.",
    },
    {
        "q": "Your team wants to use GPT-4 but needs Databricks governance and logging.",
        "a": "AI Gateway (External Models) -- create an external model endpoint for OpenAI",
        "why": "GPT-4 is not on Databricks. AI Gateway proxies the call while adding governance.",
    },
    {
        "q": "You want a code generation model with the best HumanEval score.",
        "a": "Check model cards for HumanEval benchmark -- Llama 3.3 70B scores 88.4%",
        "why": "HumanEval specifically measures code generation ability.",
    },
]

for i, q in enumerate(QUIZ, 1):
    print(f"\n  Q{i}: {q['q']}")
    print(f"  --> {q['a']}")
    print(f"  Why: {q['why']}")


**Expected output:**```============================================================================COMPLETE MODEL SELECTION FRAMEWORK============================================================================PHASE 1: DEFINE REQUIREMENTS...PHASE 2: SOURCE THE MODEL...============================================================================SELF-CHECK QUIZ: Model Selection Scenarios============================================================================  Q1: You need embeddings for a RAG pipeline. Where do you get the model?  --> Foundation Model API (databricks-bge-large-en or databricks-gte-large-en)  Why: Embedding models are available as Foundation Model APIs -- no deployment needed.  ...```

---## Exam Tips| # | Tip | Why It Matters ||---|---|---|| 1 | **Foundation Model APIs = pre-deployed, no setup.** If the exam says "immediately available" or "no endpoint creation," this is the answer. | Distinguishes from Marketplace (requires deployment) and External Models (requires API key setup). || 2 | **Marketplace "Get" does NOT deploy.** You must create a serving endpoint separately. | This is the #1 Marketplace trap. "Getting" adds to catalog; "deploying" creates the endpoint. || 3 | **External models use AI Gateway, NOT Marketplace.** OpenAI, Anthropic, Cohere are external providers. | Marketplace is for models you host on Databricks compute. External models run on the provider's infra. || 4 | **MMLU = general knowledge, HumanEval = code, MT-Bench = conversation.** Memorize these three. | The exam asks "which benchmark evaluates X?" -- these are the three most-tested. || 5 | **Bigger is not always better.** Match model size to task complexity. A fine-tuned 7B model can beat a general-purpose 70B model on a narrow task. | The exam tests cost-awareness. Over-provisioning is wrong even if it "works." || 6 | **License matters.** Apache 2.0 (Mixtral) allows unrestricted commercial use. Llama license has some restrictions. Check before deploying to production. | Exam may present a scenario where license compatibility is the deciding factor. |---## Key Takeaways### Model Sourcing on Databricks- **Foundation Model APIs**: Pre-deployed, pay-per-token, no setup. Best for prototyping and common models (Llama, DBRX, Mixtral, BGE).- **Marketplace**: Browse, compare, and deploy third-party/open-source models. Requires creating a serving endpoint after "getting" the model.- **AI Gateway (External Models)**: Proxy calls to OpenAI, Anthropic, etc. through Databricks for governance, logging, and rate limiting.### Model Card Evaluation- **Task type** must match your use case (chat, embeddings, code gen).- **Benchmarks** for shortlisting: MMLU (general), HumanEval (code), MT-Bench (chat).- **Context window** must fit your input size.- **License** must allow your intended use (commercial, redistribution, etc.).- **Always evaluate on your own data** -- benchmarks are general, your task is specific.### Decision Framework- Need it now, no setup? --> Foundation Model API- Need a specific open-source model? --> Marketplace --> Deploy to Serving- Need an external provider? --> AI Gateway (External Models)- Need the best model for your task? --> Shortlist with benchmarks, then evaluate on your data---## Next Lab**Lab 03-06: Agent Framework with MLflow** -- you will learn how to log,version, and deploy GenAI applications using MLflow. This bridges frombuilding chains (03-01) to deploying them as production endpoints.